# Projet final : les récoltes agricoles

**Équipe :** (nom de l'équipe et des 5 membres)

Une direction départementale de l'agriculture (fictive) vous confie trois ans de récoltes, de 2023 à 2025, par saison (printemps, automne), département et culture, avec la pluie tombée pendant chaque saison. Le ministère veut savoir quelles cultures pèsent le plus, où les rendements sont les meilleurs, et si la pluie explique les mauvaises années.

Ce notebook montre **votre démarche** : chaque étape dit ce que vous cherchez, le code, le résultat et ce que vous en concluez. Les graphiques doivent être visibles : avant de pousser, **Restart**, puis **Run All**.

Le tableau de bord (`app.py`) reprend ensuite vos résultats pour le décideur.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

brut = pd.read_csv("donnees/recoltes_brut.csv")

## 1. Diagnostic

Forme, types, valeurs manquantes, lignes en double, et toutes les écritures d'une même valeur dans les colonnes de texte. Notez chaque problème trouvé : vous le corrigerez à l'étape 2.

In [2]:
print("Forme :", brut.shape)
print()
print("Types :")
print(brut.dtypes)
print()
print("Valeurs manquantes :")
print(brut.isna().sum())
print()
print("Lignes en double :", brut.duplicated().sum())

Forme : (408, 6)

Types :
annee           int64
saison            str
departement       str
culture           str
surface_ha        str
production_t      str
dtype: object

Valeurs manquantes :
annee            0
saison           0
departement      0
culture          0
surface_ha       0
production_t    15
dtype: int64

Lignes en double : 18


In [3]:
for colonne in ["saison", "departement", "culture"]:
    valeurs = sorted(brut[colonne].unique())
    print(f"{colonne} : {len(valeurs)} écritures différentes")
    print(valeurs)
    print()

saison : 2 écritures différentes
['Automne', 'Printemps']

departement : 44 écritures différentes
[' Artibonite', ' Centre', " Grand'Anse", ' Nord', ' Nord-Est', ' Nord-Ouest', ' Ouest', ' Sud', ' Sud-Est', 'ARTIBONITE', 'Artibonite', 'Centre', "GRAND'ANSE", "Grand'Anse", "Grand'Anse ", 'NIPPES', 'NORD', 'NORD-EST', 'NORD-OUEST', 'Nippes', 'Nippes ', 'Nord', 'Nord ', 'Nord-Est', 'Nord-Est ', 'Nord-Ouest', 'Nord-Ouest ', 'OUEST', 'Ouest', 'Ouest ', 'SUD', 'Sud', 'Sud ', 'Sud-Est', 'Sud-Est ', 'artibonite', 'centre', "grand'anse", 'nord', 'nord-est', 'nord-ouest', 'ouest', 'sud', 'sud-est']

culture : 33 écritures différentes
[' Banane', ' Haricot', ' Igname', ' Maïs', ' Pois congo', ' Riz', ' Sorgho', 'BANANE', 'Banane', 'Banane ', 'HARICOT', 'Haricot', 'Haricot ', 'IGNAME', 'Igname', 'Igname ', 'MAÏS', 'Maïs', 'Maïs ', 'POIS CONGO', 'Pois congo', 'Pois congo ', 'Riz', 'Riz ', 'SORGHO', 'Sorgho', 'Sorgho ', 'banane', 'haricot', 'igname', 'maïs', 'pois congo', 'sorgho']



In [4]:
for colonne in ["surface_ha", "production_t"]:
    converti = pd.to_numeric(brut[colonne], errors="coerce")
    non_convertibles = brut.loc[converti.isna() & brut[colonne].notna(), colonne]
    print(f"{colonne} : {len(non_convertibles)} valeurs qui ne se convertissent pas en nombre")
    print(non_convertibles.head(8).tolist())
    print()

surface_ha : 22 valeurs qui ne se convertissent pas en nombre
['2 517', '541 ha', '1 127', '1 185', '1 185', '1 748', '492 ha', '997 ha']

production_t : 50 valeurs qui ne se convertissent pas en nombre
['3753 t', '2299 t', '1306 t', '2394 t', '599 t', '767 t', '8147 t', '2089 t']



**Les problèmes trouvés :**

- `surface_ha` et `production_t` sont lues comme du **texte** : certaines valeurs portent une unité (`541 ha`, `3753 t`) ou un espace comme séparateur de milliers (`2 517`).
- `production_t` a **15 valeurs manquantes**.
- Le fichier contient **18 lignes en double**.
- `departement` s'écrit de **44 façons** pour 10 départements, et `culture` de **33 façons** pour 7 cultures : majuscules, minuscules et espaces autour du nom.
- `saison` est propre : seulement `Printemps` et `Automne`.

## 2. Nettoyage

Une copie `propre`, puis une cellule par problème, en comptant les lignes avant et après chaque suppression. Dites ce que vous supprimez et pourquoi.

**Contrôle :** à la fin, `propre.shape` vaut `(376, 6)`, sans valeur manquante ni doublon, avec 2 valeurs dans `saison`, 10 valeurs dans `departement`, 7 valeurs dans `culture`.

In [5]:
propre = brut.copy()
print("Lignes au départ :", len(propre))

# On enlève l'unité (ha, t) et les espaces, puis on convertit en nombre
for colonne in ["surface_ha", "production_t"]:
    propre[colonne] = pd.to_numeric(propre[colonne].str.replace(r"ha|t|\s", "", regex=True), errors="coerce")

print()
print(propre[["surface_ha", "production_t"]].dtypes)
print()
print("Valeurs manquantes :")
print(propre[["surface_ha", "production_t"]].isna().sum())

Lignes au départ : 408

surface_ha        int64
production_t    float64
dtype: object

Valeurs manquantes :
surface_ha       0
production_t    15
dtype: int64


In [6]:
# Les noms officiels viennent des fichiers departements.csv et cultures.csv
departements = pd.read_csv("donnees/departements.csv")
cultures = pd.read_csv("donnees/cultures.csv")

noms_departements = {nom.lower(): nom for nom in departements["departement"]}
noms_cultures = {nom.lower(): nom for nom in cultures["culture"]}

propre["departement"] = propre["departement"].str.strip().str.lower().map(noms_departements)
propre["culture"] = propre["culture"].str.strip().str.lower().map(noms_cultures)

print("Départements :", propre["departement"].nunique(), "| noms non reconnus :", propre["departement"].isna().sum())
print("Cultures :", propre["culture"].nunique(), "| noms non reconnus :", propre["culture"].isna().sum())

Départements : 10 | noms non reconnus : 0
Cultures : 7 | noms non reconnus : 0


In [7]:
avant = len(propre)
propre = propre.drop_duplicates()
print("Lignes avant :", avant)
print("Lignes après :", len(propre))
print("Doublons supprimés :", avant - len(propre))

Lignes avant : 408
Lignes après : 390
Doublons supprimés : 18


In [8]:
avant = len(propre)
propre = propre.dropna(subset=["production_t"])
print("Lignes avant :", avant)
print("Lignes après :", len(propre))
print("Lignes sans production supprimées :", avant - len(propre))

Lignes avant : 390
Lignes après : 376
Lignes sans production supprimées : 14


In [9]:
# Sans valeur manquante, les nombres peuvent redevenir des entiers
propre["surface_ha"] = propre["surface_ha"].astype(int)
propre["production_t"] = propre["production_t"].astype(int)
propre = propre.reset_index(drop=True)

print("Forme :", propre.shape)
print("Valeurs manquantes :", propre.isna().sum().sum())
print("Doublons :", propre.duplicated().sum())
print("Saisons :", propre["saison"].nunique(), "| départements :", propre["departement"].nunique(), "| cultures :", propre["culture"].nunique())

controle = (
    propre.shape == (376, 6)
    and propre.isna().sum().sum() == 0
    and propre.duplicated().sum() == 0
    and propre["saison"].nunique() == 2
    and propre["departement"].nunique() == 10
    and propre["culture"].nunique() == 7
)
print("Contrôle réussi :", controle)

Forme : (376, 6)
Valeurs manquantes : 0
Doublons : 0
Saisons : 2 | départements : 10 | cultures : 7
Contrôle réussi : True


## 3. Jointures

Joignez `propre` avec `cultures.csv`, `pluies.csv`, et ajoutez les colonnes calculées dont vous aurez besoin. Comptez les lignes avant et après chaque `merge`.

**Contrôle :** 376 lignes après les jointures, autant qu'avant.

In [10]:
pluies = pd.read_csv("donnees/pluies.csv")

print("Lignes avant les jointures :", len(propre))

complet = propre.merge(cultures, on="culture", how="left")
print("Après la jointure avec cultures.csv :", len(complet))

complet = complet.merge(pluies, on=["annee", "saison", "departement"], how="left")
print("Après la jointure avec pluies.csv :", len(complet))

complet = complet.merge(departements, on="departement", how="left")
print("Après la jointure avec departements.csv :", len(complet))

# Colonnes calculées : le rendement (t/ha) et la valeur de la production (gourdes)
complet["rendement_t_ha"] = complet["production_t"] / complet["surface_ha"]
complet["valeur_htg"] = complet["production_t"] * 1000 * complet["prix_htg_kg"]

print()
print("Forme :", complet.shape)
print("Valeurs manquantes :", complet.isna().sum().sum())
print("Contrôle réussi :", len(complet) == 376 and complet.isna().sum().sum() == 0)

Lignes avant les jointures : 376
Après la jointure avec cultures.csv : 376
Après la jointure avec pluies.csv : 376
Après la jointure avec departements.csv : 376

Forme : (376, 15)
Valeurs manquantes : 0
Contrôle réussi : True


Enregistrez le tableau complet : c'est le fichier que lit le tableau de bord.

```python
complet.to_csv("donnees/recoltes_propre.csv", index=False)
```

In [11]:
complet.to_csv("donnees/recoltes_propre.csv", index=False)

recoltes = pd.read_csv("donnees/recoltes_propre.csv")
print("Fichier relu :", recoltes.shape)
print(recoltes.dtypes)

Fichier relu : (376, 15)
annee                     int64
saison                      str
departement                 str
culture                     str
surface_ha                int64
production_t              int64
famille                     str
rendement_moyen_t_ha    float64
prix_htg_kg             float64
pluie_mm                  int64
chef_lieu                   str
population                int64
superficie_km2            int64
rendement_t_ha          float64
valeur_htg              float64
dtype: object


## 4. Quelles cultures pèsent le plus, en tonnes puis en valeur (gourdes) ? Le classement change-t-il ?

**Ce qu'on en conclut :**

## 5. Où les rendements (tonnes par hectare) sont-ils les meilleurs, culture par culture ?

**Ce qu'on en conclut :**

## 6. La pluie explique-t-elle les rendements ? Pour quelles cultures ? Montrez-le avec des nuages de points.

**Ce qu'on en conclut :**

## 7. Quelles cultures progressent ou reculent entre 2023 et 2025 ?

**Ce qu'on en conclut :**

## 8. Rapportez la production à la population de chaque département : quels départements produisent le plus par habitant ?

**Ce qu'on en conclut :**

## 9. Conclusion

Cinq phrases au plus : ce qu'il faut retenir, ce que les données ne permettent pas de dire, ce qu'il faudrait mesurer ensuite.